# 02 · Neural networks in PyTorch — habitat threat detector

Source: `NeuralNetworks_PyTorch_Exercise`

CSV is loaded from GitHub raw first. If that fails (private repo), the notebook looks next to the original exercise folder, then asks you to upload.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Warm-up 1 — warehouse stock network

In [ ]:
warehouse_model = nn.Sequential(
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)
print(warehouse_model)

## Warm-up 2 — NumPy → tensor

In [ ]:
data = np.array([1, 2, 3, 4, 5])
tensor_data = torch.tensor(data, dtype=torch.float32)
print(tensor_data, tensor_data.dtype)

## Warm-up 3 — CrossEntropyLoss
`nn.CrossEntropyLoss` expects **raw logits**, not softmax probabilities.

In [ ]:
logits = torch.tensor(
    [[1.0, 2.0, 0.5],
     [1.5, 0.2, 1.7],
     [0.4, 0.8, 2.1]],
    dtype=torch.float32,
)
labels = torch.tensor([1, 2, 0], dtype=torch.long)
loss_fun = nn.CrossEntropyLoss()
loss = loss_fun(logits, labels)
print(f'Cross-Entropy Loss: {loss.item():.4f}')

## Habitat threat detector

In [ ]:
CSV_NAME = 'habitat_images_codebasics_DL.csv'
RAW = 'https://raw.githubusercontent.com/yash9614/lear_deepintodl/main/NeuralNetworks_PyTorch_Exercise/habitat_images_codebasics_DL.csv'
LOCALS = [
    CSV_NAME,
    f'../NeuralNetworks_PyTorch_Exercise/{CSV_NAME}',
    f'/content/{CSV_NAME}',
]

def load_csv(raw, locals_, name):
    try:
        df = pd.read_csv(raw)
        print('loaded from GitHub raw')
        return df
    except Exception as e:
        print('raw failed:', type(e).__name__)
    for p in locals_:
        if os.path.exists(p):
            print('loaded local', p)
            return pd.read_csv(p)
    try:
        from google.colab import files
        print('Upload', name)
        up = files.upload()
        return pd.read_csv(next(iter(up)))
    except Exception as e:
        raise FileNotFoundError(f'Could not load {name}') from e

df = load_csv(RAW, LOCALS, CSV_NAME)
print(df.head())
print(df.shape, df['Threat Label'].value_counts().to_dict())

In [ ]:
X = df.drop(columns=['Threat Label']).values.astype('float32')
y = df['Threat Label'].values.astype('float32')

X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-8)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y.squeeze().numpy()
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=16, shuffle=False)
print('train', len(X_train), 'test', len(X_test))

4 inputs → 8 ReLU → 1 sigmoid. BCE + Adam(lr=0.01), 20 epochs.

In [ ]:
class ThreatDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)

model = ThreatDetector().to(device)
BCE_loss = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
print(model)

In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs=20):
    model.train()
    for epoch in range(epochs):
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
        print(f'Epoch {epoch+1}/{epochs}, Loss: {running/len(train_loader):.4f}')

train_model(model, train_loader, BCE_loss, optimizer, epochs=20)

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            out = model(xb).cpu().numpy().ravel()
            preds.extend(out)
            trues.extend(yb.numpy().ravel())
    y_pred_binary = (np.array(preds) >= 0.5).astype(int)
    y_true = np.array(trues).astype(int)
    acc = accuracy_score(y_true, y_pred_binary)
    print(f'Accuracy: {acc:.2f}')
    cm = confusion_matrix(y_true, y_pred_binary)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

evaluate_model(model, test_loader)